# 06 — Option Pricing and Expiry Selection

Module 06 converts the statistical signal into an option implementation.

The Module 04 convergence horizon is treated as the **minimum required time**
for the spread-reversion thesis. It is therefore mapped to the first listed
option expiry on or after that horizon.

The strategy uses long options only:

- If \(Z_t > 0\): short the spread via a **put on the dependent stock** and a
  **call on the independent stock**.
- If \(Z_t < 0\): long the spread via a **call on the dependent stock** and a
  **put on the independent stock**.

Both options are selected at the nearest listed ATM strike.

Option prices are computed with Black-Scholes using the Module 05 EWMA
annualized volatility for each equity leg.

In [8]:
%load_ext autoreload
%autoreload 2


import numpy as np
import pandas as pd



from src.option_pricing import (
    black_scholes_price,
    trading_to_calendar_days,
    select_expiry_on_or_after,
    nearest_atm_strike,
    option_types_from_spread_direction,
    build_pair_option_trade,
)

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


## 1. Inputs

Module 06 needs:

1. Module 05 volatility output.
2. Equity prices at the evaluation date.
3. Listed option expiries.
4. Listed strikes for each underlying.
5. The risk-free rate applicable at the evaluation date.

The exact source of listed option chains and the risk-free rate can be wired
into this notebook once those data are available.

In [9]:
VOL_FILE = ("data/processed/volatility_forecasts.parquet")

PRICE_FILE = ("data/processed/train_prices.parquet")

volatility_forecasts = pd.read_parquet(
    VOL_FILE
)

prices = pd.read_parquet(
    PRICE_FILE
)

prices.index = pd.to_datetime(
    prices.index
)

prices = prices.sort_index()

volatility_forecasts.head()


,pair,dependent,independent,current_z,direction,convergence_horizon_trading_days,convergence_horizon_calendar_days,horizon_days,lambda,dependent_annualized_volatility,independent_annualized_volatility,dependent_daily_variance,independent_daily_variance,dependent_n_obs,independent_n_obs
0,SHW-HD,SHW,HD,-2.280265,-1,61,89,61,0.94,0.290474,0.273200,0.000335,0.000296,1758,1758
1,MAS-LEN,MAS,LEN,-2.414675,-1,93,135,93,0.94,0.347549,0.338222,0.000479,0.000454,1758,1758
2,SHW-DHI,SHW,DHI,-1.951851,-1,84,122,84,0.94,0.290474,0.331990,0.000335,0.000437,1758,1758
3,WMT-SPGI,WMT,SPGI,1.711591,1,83,121,83,0.94,0.200908,0.296046,0.000160,0.000348,1758,1758
4,RSG-AJG,RSG,AJG,-1.696617,-1,86,125,86,0.94,0.185075,0.213293,0.000136,0.000181,1758,1758


## 2. Evaluation-date settings

Replace these placeholders with the actual evaluation date, listed option
expiries, strikes, and risk-free rate from your data source.

In [10]:
EVALUATION_DATE = prices.index[-1]

# Placeholder only. Replace with actual listed option expiries.
AVAILABLE_EXPIRIES = pd.to_datetime(
    [
        EVALUATION_DATE + pd.Timedelta(days=30),
        EVALUATION_DATE + pd.Timedelta(days=60),
        EVALUATION_DATE + pd.Timedelta(days=90),
        EVALUATION_DATE + pd.Timedelta(days=120),
        EVALUATION_DATE + pd.Timedelta(days=150),
        EVALUATION_DATE + pd.Timedelta(days=180),
    ]
)

# Placeholder flat risk-free rate.
# Replace with the rate available at the evaluation date.
RISK_FREE_RATE = 0.03

## 3. Single-pair example

For now, strikes are illustrated with a synthetic grid around spot.
Replace these grids with actual listed strikes once the option chain is wired
into the backtest.

In [11]:
row = volatility_forecasts.iloc[0]

dep = row["dependent"]
indep = row["independent"]

dep_spot = float(
    prices.loc[:EVALUATION_DATE, dep].dropna().iloc[-1]
)

indep_spot = float(
    prices.loc[:EVALUATION_DATE, indep].dropna().iloc[-1]
)

dep_strikes = np.arange(
    max(1, np.floor(dep_spot * 0.7)),
    np.ceil(dep_spot * 1.3) + 1,
    1.0,
)

indep_strikes = np.arange(
    max(1, np.floor(indep_spot * 0.7)),
    np.ceil(indep_spot * 1.3) + 1,
    1.0,
)

trade = build_pair_option_trade(
    dependent=dep,
    independent=indep,
    direction=int(row["direction"]),
    dependent_spot=dep_spot,
    independent_spot=indep_spot,
    dependent_volatility=float(
        row["dependent_annualized_volatility"]
    ),
    independent_volatility=float(
        row["independent_annualized_volatility"]
    ),
    convergence_horizon_trading_days=int(
        row["convergence_horizon_trading_days"]
    ),
    evaluation_date=EVALUATION_DATE,
    available_expiries=AVAILABLE_EXPIRIES,
    dependent_strikes=dep_strikes,
    independent_strikes=indep_strikes,
    risk_free_rate=RISK_FREE_RATE,
)

trade

{'dependent': 'SHW',
 'independent': 'HD',
 'direction': -1,
 'required_convergence_calendar_days': 89,
 'selected_expiry': Timestamp('2023-03-27 00:00:00'),
 'selected_option_calendar_dte': 90,
 'dependent_option_type': 'call',
 'independent_option_type': 'put',
 'dependent_spot': 231.90420532226562,
 'independent_spot': 291.9278564453125,
 'dependent_strike': 232.0,
 'independent_strike': 292.0,
 'dependent_volatility': 0.29047365182092155,
 'independent_volatility': 0.2731996757783827,
 'dependent_option_price': 14.109045178034833,
 'independent_option_price': 14.712921507551698,
 'total_premium_per_share': 28.82196668558653,
 'total_premium_100x': 2882.196668558653}

## 4. Direction check

In [12]:
volatility_forecasts[
    [
        "pair",
        "direction",
        "current_z",
    ]
]

,pair,direction,current_z
0,SHW-HD,-1,-2.280265
1,MAS-LEN,-1,-2.414675
2,SHW-DHI,-1,-1.951851
3,WMT-SPGI,1,1.711591
4,RSG-AJG,-1,-1.696617
5,MCO-MSCI,-1,-1.941628
6,LYV-AXP,-1,-2.174888
7,EMR-TEL,1,3.241486
8,PNR-NWSA,-1,-1.599998


In [13]:
for _, row in volatility_forecasts.iterrows():
    dep_type, indep_type = option_types_from_spread_direction(
        int(row["direction"])
    )

    print(
        row["pair"],
        "->",
        dep_type.upper(),
        "/",
        indep_type.upper(),
    )

SHW-HD -> CALL / PUT
MAS-LEN -> CALL / PUT
SHW-DHI -> CALL / PUT
WMT-SPGI -> PUT / CALL
RSG-AJG -> CALL / PUT
MCO-MSCI -> CALL / PUT
LYV-AXP -> CALL / PUT
EMR-TEL -> PUT / CALL
PNR-NWSA -> CALL / PUT


## 5. Price every current candidate

This cell still uses placeholder expiry and strike grids. The pricing logic
is final; only the market-data wiring remains to be connected.

In [14]:
rows = []

for _, row in volatility_forecasts.iterrows():

    dep = row["dependent"]
    indep = row["independent"]

    dep_spot = float(
        prices.loc[:EVALUATION_DATE, dep].dropna().iloc[-1]
    )

    indep_spot = float(
        prices.loc[:EVALUATION_DATE, indep].dropna().iloc[-1]
    )

    dep_strikes = np.arange(
        max(1, np.floor(dep_spot * 0.7)),
        np.ceil(dep_spot * 1.3) + 1,
        1.0,
    )

    indep_strikes = np.arange(
        max(1, np.floor(indep_spot * 0.7)),
        np.ceil(indep_spot * 1.3) + 1,
        1.0,
    )

    try:
        trade = build_pair_option_trade(
            dependent=dep,
            independent=indep,
            direction=int(row["direction"]),
            dependent_spot=dep_spot,
            independent_spot=indep_spot,
            dependent_volatility=float(
                row["dependent_annualized_volatility"]
            ),
            independent_volatility=float(
                row["independent_annualized_volatility"]
            ),
            convergence_horizon_trading_days=int(
                row["convergence_horizon_trading_days"]
            ),
            evaluation_date=EVALUATION_DATE,
            available_expiries=AVAILABLE_EXPIRIES,
            dependent_strikes=dep_strikes,
            independent_strikes=indep_strikes,
            risk_free_rate=RISK_FREE_RATE,
        )

        rows.append(
            {
                "pair": row["pair"],
                "current_z": row["current_z"],
                **trade,
            }
        )

    except Exception as exc:
        print(
            f"Skipped {row['pair']}: {exc}"
        )

option_trades = pd.DataFrame(
    rows
)

option_trades

,pair,current_z,dependent,independent,direction,required_convergence_calendar_days,selected_expiry,selected_option_calendar_dte,dependent_option_type,independent_option_type,dependent_spot,independent_spot,dependent_strike,independent_strike,dependent_volatility,independent_volatility,dependent_option_price,independent_option_price,total_premium_per_share,total_premium_100x
0,SHW-HD,-2.280265,SHW,HD,-1,89,2023-03-27,90,call,put,231.904205,291.927856,232.0,292.0,0.290474,0.273200,14.109045,14.712922,28.821967,2882.196669
1,MAS-LEN,-2.414675,MAS,LEN,-1,135,2023-05-26,150,call,put,44.325928,82.781700,44.0,83.0,0.347549,0.338222,4.341074,6.719783,11.060857,1106.085682
2,SHW-DHI,-1.951851,SHW,DHI,-1,122,2023-05-26,150,call,put,231.904205,86.190697,232.0,86.0,0.290474,0.331990,18.509211,6.646152,25.155364,2515.536362
3,WMT-SPGI,1.711591,WMT,SPGI,1,121,2023-05-26,150,put,call,46.032375,304.371246,46.0,304.0,0.200908,0.296046,2.062263,24.961494,27.023757,2702.375699
4,RSG-AJG,-1.696617,RSG,AJG,-1,125,2023-05-26,150,call,put,125.125816,182.906357,125.0,183.0,0.185075,0.213293,6.745605,8.873629,15.619235,1561.923462
5,MCO-MSCI,-1.941628,MCO,MSCI,-1,141,2023-05-26,150,call,put,268.290497,441.334839,268.0,441.0,0.371192,0.382830,27.066929,40.012911,67.079841,6707.984073
6,LYV-AXP,-2.174888,LYV,AXP,-1,140,2023-05-26,150,call,put,69.589996,139.891373,70.0,140.0,0.357970,0.261306,6.562875,8.502729,15.065605,1506.560464
7,EMR-TEL,3.241486,EMR,TEL,1,177,2023-06-25,180,put,call,90.381805,107.850517,90.0,108.0,0.225365,0.292617,4.835728,9.503982,14.339711,1433.971050
8,PNR-NWSA,-1.599998,PNR,NWSA,-1,138,2023-05-26,150,call,put,42.635601,17.738493,43.0,18.0,0.323682,0.326893,3.598193,1.502530,5.100723,510.072316


## 6. Sanity checks

- Selected option DTE must be at least the required convergence calendar
  horizon.
- Premiums must be positive.
- Both legs must share the same expiry.

In [15]:
assert (
    option_trades[
        "selected_option_calendar_dte"
    ]
    >=
    option_trades[
        "required_convergence_calendar_days"
    ]
).all()

assert (
    option_trades[
        "dependent_option_price"
    ] > 0
).all()

assert (
    option_trades[
        "independent_option_price"
    ] > 0
).all()

## 7. Save Module 06 output

In [16]:
OUTPUT = ("data/processed/option_trades.parquet"
)

option_trades.to_parquet(
    OUTPUT,
    index=False,
)

print(
    f"Saved: {OUTPUT}"
)

Saved: data/processed/option_trades.parquet


## Important before the final backtest

The placeholder expiry list, strike grids, and flat risk-free rate in this
notebook are only scaffolding.

The next step is to connect the actual historical option-chain information
available to the project. The functions in `option_pricing.py` are already
designed to accept real listed expiries and strikes directly.